# Reproducible Data Cleaning with Pandas

*Session 3 of the workshop series **Reproducible Research with Python***

This notebook introduces tools for data cleaning in pandas. By the end of this notebook, you will be able to:
- Save a raw copy of a dataset before changing anything
- Drop and rename columns
- Create new columns from existing ones using math, string methods, and date-time conversion
- Subset a dataset by a condition and save the cleaned result
- Combine all of these steps into a single reproducible workflow that you can rerun on new data at any time

## The "Excel Problem" of Data Cleaning

If you've done data cleaning before, you likely worked in a spreadsheet software like Excel or Google Sheets. While these platforms make it easy to view all your data at once and easily make changes to it, they introduce many opportunities for problems, such as human error in correcting data in individual cells or formatting issues with different data types.

But most importantly, **data cleaning in Excel isn't easily reproducible.** You don't get any record of the ways that you transformed your dataset (unless you write them down yourself), and if you need to clean a different dataset with the same structure, you will need to manually repeat all your steps. That typically won't be a problem for small datasets or cleaning workflows with only a few steps, but it quickly becomes tedious at best and error-prone at worst for more complicated scenarios.

This is where pandas comes in. **If you can write a script in Python for cleaning your data, then you can clean your data in the exact same way an infinite number of times.** Need to make a change to your workflow? Not a problem! You can just make the necessary change to your code and rerun it, knowing that all the other pieces of the process don't need to be changed. At the end of this session, we'll see this payoff firsthand.

## Importing Libraries

As we did last time, we start by importing the libraries that we will need to use in this notebook:

In [ ]:
import pandas as pd
import requests

## Review: Loading the Earthquake Data

Last session, we downloaded a week of [earthquake data from the US Geological Survey](https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php). That feed updates regularly, so the data has changed since then. We'll start by loading in this week's data similar to our code from last week.

This time, we'll also grab one extra piece of information. Each earthquake's `"geometry"` contains its coordinates as a **list** in the form `[longitude, latitude, depth]`, with depth measured in kilometers. We can collect the depths with a second `for` loop (grabbing the item at index `2`) and attach them to the DataFrame by assigning the list to a column name that doesn't exist yet — pandas will create the new column for us:

In [ ]:
# Download the past 7 days of earthquakes:
url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_week.geojson"
data = requests.get(url).json()

# Build a list of each earthquake's properties:
quake_list = []
for feature in data["features"]:
    quake_list.append(feature["properties"])

quakes = pd.DataFrame(quake_list)

# Collect each earthquake's depth (the third coordinate, at index 2):
depth_list = []
for feature in data["features"]:
    depth_list.append(feature["geometry"]["coordinates"][2])

# Assign the list to a new column in the DataFrame:
quakes["depth_km"] = depth_list

quakes.head()

Try writing your own code to answer the following questions about the `quakes` dataset (or just copy it from the previous lesson's notebook):
- How many rows and columns does it contain?
- What are the names of the columns?
- How many times does each value occur in the `"type"` column?
- What was the strongest seismic event this week?
- How many events had a magnitude of 4.5 or greater?

In [ ]:
# Explore the dataset here.
# You can write code for each answer in an individual cell,
# or you can wrap your code inside a print() statement
# so you can output all information in a single cell.


## Saving the Raw Data

Before we make any changes to our dataset, we should make sure that we save a raw copy of it. This copy will:
- Make sure we can restore the original version of the dataset if we make a mistake
- Serve as a point of comparison for our cleaned and transformed dataset

(This is especially important for live data: the feed will have moved on by tomorrow, so this file is our only record of exactly what we downloaded today.)

Recall that to save a DataFrame to a file, we can use the `.to_csv()` method of a DataFrame:

In [ ]:
# Save the raw quakes DataFrame to a csv:
quakes.to_csv("quakes_raw.csv", index=False)

## Reproducible Data Cleaning and Transformation in Pandas

We typically need to make some changes to datasets before we are able to use them. For our earthquake data, we will want to:
- Delete some extraneous columns
- Rename columns to make them clearer
- Create new columns (from numbers, text strings, and date-time data)
- Filter the data by some conditions

All or some of these may be tasks you're familiar with doing in Excel, and the point-and-click interface may feel more intuitive. But this dataset has thousands of rows, and a new version of it exists every minute. Cleaning it by hand once would be tedious; cleaning it by hand every time you wanted fresh data would be impossible.

**By working in pandas, we can create a single data cleaning workflow, and then run it any number of times on data of the same structure.** Setting up the initial data cleaning process takes more work, but it will save time and prevent mistakes in the future.

We'll now go over how to clean our `quakes` DataFrame with the steps above, and will then put them together into a single workflow. These are by no means all the data cleaning steps you might take in pandas. The goal here isn't to make you an expert but get you thinking about how individual steps come together into a reproducible workflow.

### Deleting Extraneous Columns with `.drop()`

We aren't interested in everything in our `quakes` DataFrame. To declutter it, we can remove columns using the `.drop()` method, in the format `quakes.drop("columnName", axis=1)`.

By default, `.drop()` will remove a **row**, not a **column**, from a DataFrame. For example, `quakes.drop(0)` would drop the first row (because its index is 0). Since we want to remove columns instead, we need to add in `axis=1`. This tells Python to look for the specified name (or names) in the columns of the DataFrame, not the index.

The `"tz"` column is a leftover from an older version of the USGS feed and contains no information, so let's drop it from the DataFrame:

In [ ]:
quakes.drop("tz", axis=1)

<h3 style="color:red; display:inline">Assign Your DataFrame to Its Variable Name as You Update It!</h3>

`quakes.drop("tz", axis=1)` returned (and displayed) a new DataFrame without `"tz"`, but that doesn't mean the column is gone from our `quakes` DataFrame.

Why not? **Because we didn't change the `quakes` variable.** If we want a change to our DataFrame (such as dropping a column) to be permanent, we need to assign the changed DataFrame to a variable:

In [ ]:
# Print the shape of the quakes DataFrame before we change it:
print(quakes.shape)

# Drop the tz column, saving the changed DataFrame to the quakes variable:
quakes = quakes.drop("tz", axis=1)

# Print the new shape of quakes:
print(quakes.shape)

Recall that the `.shape` attribute returns `(number of rows, number of columns)`, so we can see that our column was successfully dropped.

### The Test-Update-Repeat Workflow

A common workflow for data cleaning in pandas is to:
1. Write code to make a change to a DataFrame.
2. Examine the changed DataFrame to make sure it's been changed as intended.
3. Edit the code to update the variable for the DataFrame with the transformed version.

What if you make a mistake? Don't worry! Just remove your code that caused the mistake, rerun the code that loads your data to restore the original version, and then run the rest of your working code. That ability to unwind mistakes by rerunning a few lines of code is a key advantage of pandas over Excel.

(This also explains an error you may hit today: if you run a `.drop()` cell a second time, pandas will complain that the column doesn't exist — because you already dropped it!)

### Dropping Multiple Columns at Once

`.drop()` can also take a **list** of column names. Several columns in this dataset contain URLs and codes meant for computers rather than people, so let's remove a whole batch of them. Since we've already practiced the test-update-repeat pattern, this time we'll assign the result directly and confirm the change afterwards by checking `.columns`:

In [ ]:
columns_to_drop = ["url", "detail", "ids", "sources", "types", "net", "code"]

quakes = quakes.drop(columns_to_drop, axis=1)

# Examine the columns in quakes and confirm that the ones we wanted to drop have
# been removed:
quakes.columns

In [ ]:
# Exercise: The "nst", "gap", "dmin", and "rms" columns are technical
# measurements of detection quality that we won't use. Drop all four
# in a single line of code and save the result to quakes:


### Renaming a Column with `.rename()`

The `"mag"` column name is a little cryptic, so let's rename it to `"magnitude"`.

To do so, we use `.rename()` in the form `quakes.rename({"currentName" : "newName"}, axis=1)`. Notice that the pairing of old and new name is a **dictionary**. The current name is the key, and the new name is its value. Again, we need to specify `axis=1` so that pandas works with the columns of our DataFrame, not the index.

In [ ]:
quakes.rename({"mag" : "magnitude"}, axis=1).head()

Does the DataFrame look right to you? If so, save the updated version to the `quakes` variable. Then, repeat the process to rename the `"sig"` column to `"significance"`.

In [ ]:
# Rename the "mag" column and save the changed version to quakes:
quakes = quakes.rename({"mag" : "magnitude"}, axis=1)

# Exercise: Rename the "sig" column to "significance". Once you have checked
# that the change was made as intended, save it to the quakes variable:


### Creating a New Column from Math on an Existing One

Pandas makes it easy to transform all the data in a column at once. The simplest way to do this is with mathematical operators.

For example, we could write code to return double the value of every magnitude:

In [ ]:
quakes["magnitude"] * 2

Remember: a single column in a DataFrame is a **Series**. When we perform a mathematical operation such as multiplication (`*`) on a Series, that operation gets performed on each value and returns a new Series containing those values. It works like the `for` loops we worked with previously, but pandas builds the loop for us under the hood.

Let's try a genuinely useful conversion. Our `"depth_km"` column is in kilometers; for a US audience, miles may be more familiar. One kilometer is about 0.621 miles:

In [ ]:
quakes["depth_km"] * 0.621

To save this data to a new column, we call the `quakes` DataFrame with the new column name we'd like to use in brackets — the same move we used to attach `"depth_km"` at the start of this notebook:

In [ ]:
quakes["depth_miles"] = quakes["depth_km"] * 0.621

In [ ]:
# Exercise: Try writing code to view the "depth_km" and "depth_miles"
# columns side by side:


### Creating a New Column with String Methods

Math operators aren't the only thing that works across a whole column at once. The string methods from Session 1 (like `.lower()`) have column-wide versions too, accessed by putting `.str` between the column and the method.

The `"place"` column contains descriptions like `"8 km ENE of Pahala, Hawaii"`. Suppose we want just the region — the part after the comma — so that we can count earthquakes by region. On a single string, we could use the `.split()` method, which divides a string into a **list** of pieces wherever a separator (in this case, `", "`) occurs:

In [ ]:
"8 km ENE of Pahala, Hawaii".split(", ")

The region is the **last** item in the resulting list, which we can grab with the negative index `[-1]`. To apply this to every value in the column at once, we put `.str` before each string operation:

In [ ]:
quakes["place"].str.split(", ").str[-1]

Look closely at the results: not every place description contains a comma (for example, `"south of the Fiji Islands"`). When there's nothing to split, we simply get the whole string back, but we don't get an error or warning. This makes examining your data after each transformation important to catch anything that didn't go as planned.

Let's save the result as a new column, then use `.value_counts()` to see which regions had the most seismic activity this week and note any region names that didn't split properly:

In [ ]:
quakes["region"] = quakes["place"].str.split(", ").str[-1]

# Temporarily increase the number of rows that get displayed:
with pd.option_context('display.max_rows', None):
    print(quakes["region"].value_counts())

In [ ]:
# Exercise: Use .str.split() to display the specific place and region name from
# the "place" column (removing the distance such as "8 km ENE"). Examine some
# of the values of "place" to determine what separator to use.
# (Just display these values; no need to save the result.)


## Reformatting Date and Time Data

Date and time information (or time-series data) is often an important part of any dataset, but it can be frustrating to work with. Even in pandas, the ins and outs of working with this type of data can be confusing. This section won't make you an expert, but it will demonstrate the flexibility and control that pandas can offer you if you work with time and date data frequently.

In this dataset, the `"time"` column is not easily interpretable:

In [ ]:
quakes["time"].head()

These giant integers are Unix timestamps. We can see in the (data documentation)[https://earthquake.usgs.gov/data/comcat/index.php#time] that they represent the number of milliseconds that have elapsed since midnight on January 1, 1970. This format is great for computers to be able to compare different time values in absolute form, but it's less helpful for humans.

This level of detail is also more than we need. For example, what if we wanted to group our earthquakes by day rather than by millisecond? We would need dates (without additional time information) in their own column.

Pandas makes it possible to consistently convert date-time information into other formats, but the process takes several steps.

### Step 1: Convert the column to Timestamp objects

Pandas has a special data type, the Timestamp, which can be manipulated as date-time information. To convert our column, we use the `to_datetime()` function provided by pandas, telling it that our numbers are in units of milliseconds with `unit="ms"`.

We'll assign these converted values (stored as a Series) to a variable `times`:

In [ ]:
# Convert the values in "time" from Unix timestamps to pandas Timestamps:
times = pd.to_datetime(quakes["time"], unit="ms")

# Check the type of the new values contained in the times Series:
type(times[0])

In [ ]:
# Display the times Series:
times

Those times are already vastly more readable, but we wouldn't be able to use them to group quakes by date quite yet. If we check to see how many unique values there are, we will see that every time is different (which makes sense, as they're stored down to the millisecond):

In [ ]:
len(times.unique())

### Step 2: Reformat data to display in YYYY-MM-DD format

Now that we have converted our data to Timestamps, we can convert it once again to a string containing only the relevant information. In our case, we will preserve only the year, month, and day, but not the hour, minute, or second.

To do so, we use the method `.dt.strftime()`, passing in a string that indicates how we would like the resulting data to be displayed. We will use `"%Y-%m-%d"` to display first the year in four digits, the month in two digits, and the date in two digits:

In [ ]:
times.dt.strftime("%Y-%m-%d").unique()

This `%` notation is very flexible and allows you to rewrite dates according to your needs, including reordering the information or adding in time in addition to date. Remembering all the date and time codes isn't intuitive, but you can find a complete list of them in the [Python documentation](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes).

If we wanted to, we could change the order of date information to the US convention of month-day-year:

In [ ]:
times.dt.strftime("%m-%d-%Y")

...or even add in the day of the week:

In [ ]:
times.dt.strftime("%a %m-%d-%Y")

Using the [Python documentation](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes), try experimenting with some other date-time information you can convey:

In [ ]:
# Try formatting your own date-time values here:


### Step 3: Create a new column for date values

Now that we know how to derive date information from our Timestamps, we can create a new column called `"date"` and populate it with our date values — the same column-creation move we've now used twice:

In [ ]:
quakes["date"] = times.dt.strftime("%Y-%m-%d")

We can now view the new `"date"` column in our `quakes` DataFrame:

In [ ]:
quakes.head()

In [ ]:
# Exercise: Confirm that the "date" column has fewer unique
# values than the "time" column does using unique():

In [ ]:
# Exercise: Which date this week had the most seismic events?
# (Hint: you learned the method for counting values last session.)


## Subsetting by Condition

A common data cleaning task is to reduce a dataset to only include rows that meet a certain condition. The weekly feed includes thousands of tiny tremors that only sensitive instruments can detect. Earthquakes of magnitude 2.5 and above are roughly the ones that people can actually feel near the epicenter, so let's reduce our dataset to those.

Recall from last session that we create a **mask** by writing a **comparison operator** expression to check which rows in a column meet a given condition, and then pass that mask to our DataFrame:

In [ ]:
# Step 1: Create a mask for the condition we want to use to subset our data:
mask = quakes["magnitude"] >= 2.5

# Step 2: Pass the mask to the original DataFrame:
quakes[mask]

This time, let's take the additional step of assigning our subsetted DataFrame to its own variable:

In [ ]:
felt_quakes = quakes[mask]

In [ ]:
# Exercise: How many earthquakes this week were magnitude 2.5 or greater?


In [ ]:
# Exercise: Create a dataframe containing earthquakes of magnitude 2.5 or
# greater that occurred on Monday:


## Saving the Transformed Dataset

We've now made a number of changes to the data we originally accessed from the USGS. We've dropped columns we don't need, renamed columns with unclear names, derived new columns using math (`"depth_miles"`), string methods (`"region"`), and date-time conversion (`"date"`), and finally subset our DataFrame to only earthquakes people could feel.

To round things off, let's save our transformed dataset to a new CSV. Note that we're using a **different file name** than `quakes_raw.csv`. Our raw copy stays untouched so we can return to it in the future:

In [ ]:
felt_quakes.to_csv("quakes_clean.csv", index=False)

## Putting It All Together: Building a Reproducible Workflow

We now have a series of steps that will allow us to transform a dataset according to our specifications. Let's put it all together into a single code block.

Review the code above and copy the portions into the cell below that will:
1. Load the data from the USGS feed (including the depth column)
2. Save the raw data to `quakes_raw.csv`
3. Drop the `"tz"` column and the other extraneous columns
4. Rename `"mag"` to `"magnitude"` and `"sig"` to `"significance"`
5. Create the `"depth_miles"` column
6. Create the `"region"` column from `"place"`
7. Convert `"time"` to Timestamps and create the `"date"` column
8. Subset to earthquakes of magnitude 2.5 or greater
9. Save the final dataset to `quakes_clean.csv`

In [ ]:
# Bring together the full workflow here:


## Rerunning Your Workflow on New Data

Here is where writing code really pays off. The USGS publishes several feeds at nearly identical URLs:

|Feed|URL ending|
|---|---|
|Past day|`all_day.geojson`|
|Past 7 days|`all_week.geojson`|
|Past 30 days|`all_month.geojson`|
|Magnitude 4.5+, past 30 days|`4.5_month.geojson`|

Change the URL in your workflow above to the past 30 days and rerun the cell. Every step happens again instantly, on several times as much data. Cleaning a month of earthquake data by hand in Excel would be tedious, but you now have the tools to do it in seconds with a complete written record of exactly what was done.

And if a collaborator (or a peer reviewer!) asks how you prepared your data, your Jupyter notebook provides the documentation you need.

In [ ]:
# Exercise: Copy your workflow from the cell above, change the URL to the
# past 30 days, and save the results to quakes_month_raw.csv and
# quakes_month_clean.csv (so you don't overwrite this week's files!):


# Extending Your Pandas Knowledge with AI Coding Tools

If you'd like, apply your pandas knowledge to work with your AI tool of choice to try writing code for the following tasks with the `felt_quakes` data. Rather than detail your specific dataset in the prompt, try formulating the task in general terms using pandas terminology and adapting the generalized code you receive to your data:
1. Display summary statistics (such as mean and standard deviation) for each `region`.
2. Most state names in `region` are written out with their full name, but "California" is abbreviated to "CA". Change the values of "CA" in the `region` column to "California".
3. Create a bar chart showing the number of felt quakes on each day.
4. Investigate if there's a relationship between magnitude of a quake and depth by creating a scatterplot.
5. The `felt` column contains the number of reports submitted to the USGS's ["Did You Feel It?" system](https://earthquake.usgs.gov/data/dyfi/). What variables might have a relationship to the number of submitted reports? Try grouping your data by some variable or generating a scatterplot to investigate that relationship.

In [ ]:
# Try out the exercises above here (and create more cells below):


# Continue Your Pandas Journey

This lesson has given you a taste of the power of pandas, but it's only scratched the surface of how you can automate and reproduce your data cleaning workflows. For the most thorough overview of everything pandas has to offer, see the e-book [Python for Data Analysis (3rd Edition)](https://wesmckinney.com/book/) by Wes McKinney, the creator of pandas.

For tutorial-based learning, download the following notebooks and run them in Jupyter:
- **Pandas Basics 1:** https://github.com/ithaka/constellate-notebooks/blob/master/Pandas-basics/pandas-basics-1.ipynb
- **Pandas Basics 2:** https://github.com/ithaka/constellate-notebooks/blob/master/Pandas-basics/pandas-basics-2.ipynb
- **Pandas Basics 3:** https://github.com/ithaka/constellate-notebooks/blob/master/Pandas-basics/pandas-basics-3.ipynb
- **Pandas Intermediate 1:** https://github.com/ithaka/constellate-notebooks/blob/master/Python-intermediate/python-intermediate-1.ipynb
- **Pandas Intermediate 2:** https://github.com/ithaka/constellate-notebooks/blob/master/Python-intermediate/python-intermediate-2.ipynb
- **Pandas Intermediate 3:** https://github.com/ithaka/constellate-notebooks/blob/master/Python-intermediate/python-intermediate-3.ipynb
- **Pandas Intermediate 4:** https://github.com/ithaka/constellate-notebooks/blob/master/Python-intermediate/python-intermediate-4.ipynb
- **Pandas Intermediate 5:** https://github.com/ithaka/constellate-notebooks/blob/master/Python-intermediate/python-intermediate-5.ipynb

# Acknowledgments
<img align="left" src="https://ithaka-labs.s3.amazonaws.com/static-files/images/tdm/tdmdocs/CC_BY.png"><br />

This lesson was created by Isaac Wink and is shared under a [Creative Commons CC BY License](https://creativecommons.org/licenses/by/4.0/).<br />